### Qualitative variables to quantitative

- export each class to a single file
- run focal statistics for each new raster

In [ ]:
ivars = [
    #'X://atrisk_fire1/models_centro/aspect.tif',
    'X://susmod_tk2/lulc/COS1995v2.tif',
    'X://susmod_tk2/lulc/COS2007v3.tif',
    'X://susmod_tk2/lulc/COS2010v2.tif',
    'X://susmod_tk2/lulc/COS2015v2.tif'
]

refclip = 'X://susmod_tk2/roi_ptctr.shp'

refrst = 'X://susmod_tk2/dem_roi.tif'

csize = 10

clsrst = 'X://susmod_tk2/lulctmp'

fvars = 'X://susmod_tk2/lulcfeat'

In [ ]:
import numpy as np
import os
import arcpy

from glass.esri.rd.rst import rst_to_array
from glass.esri.prop.rst import get_nodata, rst_geoprop
from glass.pys.oss import fprop
from glass.esri.wt import obj_to_rst
from glass.esri.rst.neigh import focal_statistics
from glass.esri.rst.dist import euclidean_distance

In [ ]:
arcpy.env.outputCoordinateSystem = ivars[0]

In [ ]:
arrays, clss = [], []
rprop = []
for r in ivars:
    array = rst_to_array(r)
    ndval = get_nodata(r)
    gprop = rst_geoprop(r)
    
    unique = np.unique(array)
    
    arrays.append(array)
    clss.append(list(unique[unique != ndval]))
    rprop.append(gprop)

In [ ]:
print(ndval)

In [ ]:
clss

In [ ]:
# Create a new raster for each class in the data

for i in range(len(ivars)):
    fname = fprop(ivars[i], 'fn')
    
    for cls in clss[i]:
        acls = np.zeros(arrays[i].shape, dtype=arrays[i].dtype)
        
        np.place(acls, arrays[i] == cls, 1)
        
        orst = obj_to_rst(
            acls,
            os.path.join(clsrst, f'{fname}_{str(cls)}.tif'),
            rprop[i][0], rprop[i][1], 0
        )
        
        # Run focal statistics
        #focal_statistics(
            #orst,
            #os.path.join(fvars, f'{fname}_{str(cls)}.tif'),
            #"Rectangle", focal_rectangle, "SUM"
        #)
        
        euclidean_distance(
            orst, csize,
            os.path.join(fvars, f'{fname}_{str(cls)}.tif'),
            template=refrst,
            boundary=refclip,
            snap=refrst
        )
        